# Image Color Mood Analyzer - Colab Demo

This notebook demonstrates the same functionality as the Hugging Face Space.
Upload any image and get its dominant color palette with emotional associations based on color psychology.

**No ML model needed** - just image processing with PIL and KMeans clustering.

In [ ]:
!pip install -q Pillow scikit-learn numpy matplotlib

In [ ]:
import numpy as np
import colorsys
from PIL import Image
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from google.colab import files
from IPython.display import display, HTML
import io

In [ ]:
MOOD_TABLE = [
    (15,  "Passionate", "Fiery energy and intensity"),
    (45,  "Warm",       "Comfort and enthusiasm"),
    (75,  "Joyful",     "Optimism and brightness"),
    (160, "Harmonious", "Growth and balance"),
    (200, "Tranquil",   "Clarity and peace"),
    (260, "Calm",       "Depth and trust"),
    (290, "Mysterious", "Intrigue and spirituality"),
    (345, "Romantic",   "Tenderness and playfulness"),
    (361, "Passionate", "Fiery energy and intensity"),
]

def color_to_mood(r, g, b):
    h, l, s = colorsys.rgb_to_hls(r / 255, g / 255, b / 255)
    hue = h * 360
    if s < 0.15:
        return ("Neutral", "Understated and balanced")
    for threshold, mood, desc in MOOD_TABLE:
        if hue < threshold:
            if l < 0.25:
                mood = "Deep " + mood
            elif l > 0.75:
                mood = "Light " + mood
            return (mood, desc)
    return ("Neutral", "Balanced tone")

def analyze_image(image):
    img = image.resize((100, 100)).convert("RGB")
    pixels = np.array(img).reshape(-1, 3).astype(float)
    mask = (pixels.sum(axis=1) > 50) & (pixels.sum(axis=1) < 700)
    pixels = pixels[mask] if mask.sum() > 10 else pixels

    km = KMeans(n_clusters=5, n_init=10, random_state=42)
    km.fit(pixels)
    colors = km.cluster_centers_.astype(int)
    counts = np.bincount(km.labels_)
    order = counts.argsort()[::-1]

    return colors, counts, order

## Upload an Image

Run the cell below, click "Choose Files", and select any image.

In [ ]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]
image = Image.open(io.BytesIO(uploaded[filename]))
print(f"Loaded: {filename} ({image.size[0]}x{image.size[1]})")
display(image.resize((300, int(300 * image.size[1] / image.size[0]))))

## Analyze Color Mood

In [ ]:
colors, counts, order = analyze_image(image)

# Build HTML output similar to the Space
bar_html = '<div style="display:flex;height:28px;border-radius:8px;overflow:hidden;border:1px solid #e0e0e0;">'
for i in order:
    r, g, b = colors[i]
    pct = counts[i] / counts.sum() * 100
    bar_html += f'<div style="width:{pct}%;background:rgb({r},{g},{b});min-width:4px;"></div>'
bar_html += '</div>'

cards_html = ''
for i in order:
    r, g, b = colors[i]
    pct = counts[i] / counts.sum() * 100
    mood, desc = color_to_mood(r, g, b)
    cards_html += f'''
    <div style="display:flex;align-items:center;gap:14px;padding:10px 14px;border-radius:10px;
                background:#fafafa;border:1px solid #eee;margin:8px 0;">
        <div style="width:48px;height:48px;border-radius:8px;background:rgb({r},{g},{b});
                    border:1px solid rgba(0,0,0,0.08);flex-shrink:0;"></div>
        <div>
            <div style="font-weight:700;font-size:1.05em;">{mood}</div>
            <div style="color:#666;font-size:0.85em;margin-top:2px;">{desc}</div>
            <div style="color:#aaa;font-size:0.78em;margin-top:3px;font-family:monospace;">rgb({r},{g},{b}) - {pct:.0f}%</div>
        </div>
    </div>'''

display(HTML(f'<div style="font-family:system-ui;max-width:500px;">{bar_html}<div style="margin-top:12px;">{cards_html}</div></div>'))

## Matplotlib Visualization

An alternative view using matplotlib to show the color palette as a chart.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Palette bar
left = 0
for i in order:
    r, g, b = colors[i]
    pct = counts[i] / counts.sum()
    mood, _ = color_to_mood(r, g, b)
    ax1.barh(0, pct, left=left, color=(r/255, g/255, b/255), edgecolor='white', height=0.6)
    if pct > 0.1:
        ax1.text(left + pct/2, 0, f'{pct:.0%}', ha='center', va='center', fontsize=9, fontweight='bold',
                color='white' if (r+g+b)/3 < 128 else 'black')
    left += pct
ax1.set_xlim(0, 1)
ax1.set_yticks([])
ax1.set_title('Color Proportions')

# Color swatches with mood labels
for idx, i in enumerate(order):
    r, g, b = colors[i]
    mood, _ = color_to_mood(r, g, b)
    rect = patches.Rectangle((0, 4-idx-0.4), 0.8, 0.8, facecolor=(r/255, g/255, b/255), edgecolor='gray')
    ax2.add_patch(rect)
    ax2.text(1.0, 4-idx, f'{mood}  (rgb {r},{g},{b})', va='center', fontsize=10)
ax2.set_xlim(-0.2, 5)
ax2.set_ylim(-0.5, 5)
ax2.set_yticks([])
ax2.set_xticks([])
ax2.set_title('Color Moods')

plt.tight_layout()
plt.show()